### **failure rate analysis**

In [1]:
import pandas as pd

raw_df = pd.read_csv(
    "./data/payment_transactions_raw.csv",
    parse_dates = ["transaction_date"]
)

clean_df = raw_df.copy()
clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   transaction_id     30000 non-null  object        
 1   customer_id        30000 non-null  object        
 2   merchant_id        30000 non-null  object        
 3   transaction_date   30000 non-null  datetime64[ns]
 4   payment_method     30000 non-null  object        
 5   amount             30000 non-null  float64       
 6   channel            30000 non-null  object        
 7   merchant_category  30000 non-null  object        
 8   status             30000 non-null  object        
 9   failure_reason     3663 non-null   object        
dtypes: datetime64[ns](1), float64(1), object(8)
memory usage: 2.3+ MB


### **failure rate by channel**

In [2]:
clean_df["is_failed"] = clean_df["status"].eq("Failed")

failure_rate_by_channel = (
    clean_df
    .groupby("channel")
    .agg(
        tranx_count = ("transaction_id", "nunique"),
        failed_tranx = ("is_failed", "sum")
    )
)

failure_rate_by_channel["failure_rate"] = (failure_rate_by_channel["failed_tranx"] / failure_rate_by_channel["tranx_count"]).round(4)
failure_rate_by_channel.to_csv("./output/failure_rate_by_channel.csv")
failure_rate_by_channel

,tranx_count,failed_tranx,failure_rate
channel,,,
Mobile App,13453,1623,0.1206
POS,5993,639,0.1066
Web,10554,1401,0.1327


### **channel's share of total failures**

In [4]:
channel_share_of_total_failures = (
    clean_df
    .groupby("channel")
    .agg(
        channel_failed_tranx = ("is_failed", "sum")
    )
)

channel_share_of_total_failures["Failed Transactions"] = channel_share_of_total_failures["channel_failed_tranx"].sum()
channel_share_of_total_failures["Channel's Share of Total Failures"] = (channel_share_of_total_failures["channel_failed_tranx"] / channel_share_of_total_failures["Failed Transactions"]).round(4)
channel_share_of_total_failures.to_csv("./output/channel_share_of_total_failures.csv")
channel_share_of_total_failures

,channel_failed_tranx,Failed Transactions,Channel's Share of Total Failures
channel,,,
Mobile App,1623,3663,0.4431
POS,639,3663,0.1744
Web,1401,3663,0.3825


### **failure rate by payment method**

In [3]:
failure_rate_by_payment_method = (
    clean_df
    .groupby("payment_method")
    .agg(
        tranx_count = ("transaction_id", "nunique"),
        failed_tranx = ("is_failed", "sum")
    )
)

failure_rate_by_payment_method["failure_rate"] = (failure_rate_by_payment_method["failed_tranx"] / failure_rate_by_payment_method["tranx_count"]).round(4)
failure_rate_by_payment_method.sort_values("failure_rate", ascending=False)
failure_rate_by_payment_method.to_csv("./output/failure_rate_by_payment_method.csv")
failure_rate_by_payment_method

,tranx_count,failed_tranx,failure_rate
payment_method,,,
Bank Transfer,4439,648,0.1460
Credit Card,9524,996,0.1046
Debit Card,8409,1067,0.1269
Mobile Wallet,7628,952,0.1248


### **payment methods share of total failures**

In [6]:
payment_method_share_of_total_failures = (
    clean_df
    .groupby("payment_method")
    .agg(
        failed_tranx = ("is_failed", "sum")
    )
)

payment_method_share_of_total_failures
payment_method_share_of_total_failures["Failed Transactions"] = payment_method_share_of_total_failures["failed_tranx"].sum()
payment_method_share_of_total_failures["Payment Method Share of Total Failures"] = (payment_method_share_of_total_failures["failed_tranx"] / payment_method_share_of_total_failures["Failed Transactions"]).round(4)
payment_method_share_of_total_failures.to_csv("./output/payment_method_share_of_total_failures.csv")
payment_method_share_of_total_failures

,failed_tranx,Failed Transactions,Payment Method Share of Total Failures
payment_method,,,
Bank Transfer,648,3663,0.1769
Credit Card,996,3663,0.2719
Debit Card,1067,3663,0.2913
Mobile Wallet,952,3663,0.2599
